<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 07. GIN: 합 집계와 WL 표현력

## How Powerful are Graph Neural Networks?

- **원 논문:** [How Powerful are Graph Neural Networks?](https://openreview.net/forum?id=ryGs6iA5Km)
- **저자 / 발표:** Keyulu Xu, Weihua Hu, Jure Leskovec, Stefanie Jegelka · ICLR (2019)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** message passing, multiset, Weisfeiler–Lehman test
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** sum aggregator와 (1+epsilon) self term을 구현해 mean이 잃는 degree 정보를 보존함을 확인합니다.

**축소하거나 생략한 부분:** 9개 graph-classification benchmark 대신 한 로컬 그래프의 node multiset 표현력을 점검합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §4.1, Lemma 5 및 Corollary 6 | injective multiset sum 직관 | degree 구별 |
| §4.2, Eq. (4.1) | GIN update | epsilon self term |
| §4.3, Eq. (4.2) | layer별 graph readout | permutation invariance |
| Figure 2 | mean/max/sum 표현력 비교 | 고유 representation 수 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all,Path,np.load,torch.tensor,torch.float32,torch.bool,torch.allclose,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 11개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$h_v^k=\operatorname{MLP}^k\!\left((1+\epsilon^k)h_v^{k-1}+\sum_{u\in
\mathcal N(v)}h_u^{k-1}\right)$$

- **기호 정의:** ε은 self 가중치, 합은 이웃 multiset, MLP는 injective map 근사입니다.
- **수식의 역할:** sum aggregation으로 mean이 지우는 이웃 중복도와 degree 정보를 보존합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** 두 GIN layer가 §4.2의 update를 수행하고,
  layer-wise sum readout과 graph prediction head가 §4.3을 구현합니다.
- **입출력 shape:** ego graph [N_g, F], [N_g, N_g] →
  layer states → concatenated graph readout → graph logit [1, C]
- **평가:** held-out ego-graph accuracy, readout 순열 불변성,
  sum/mean의 degree·multiset 분리
- **원문 대비 한계:** 한 로컬 graph의 ego-network를 graph sample로 사용하므로,
  원 논문의 9개 graph-classification benchmark와 직접 비교하지 않습니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §4.2, Eq. (4.1)
- **이 셀의 책임:** GIN update
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** epsilon self term. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.ones -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def gin_aggregate(x: Tensor, adj: Tensor, epsilon: float = 0.0) -> Tensor:
    return (1 + epsilon) * x + adj @ x


ones = torch.ones(len(adjacency), 1)
gin_counts = gin_aggregate(ones, adjacency)
assert torch.allclose(gin_counts[:, 0], adjacency.sum(1) + 1)

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §4.1, Lemma 5 및 Corollary 6 / Figure 2
- **이 셀의 책임:** injective multiset sum 직관 / mean/max/sum 표현력 비교
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** degree 구별 / 고유 representation 수. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
degree = adjacency.sum(1, keepdim=True)
mean_neighbors = (adjacency @ ones) / degree.clamp_min(1)
active = degree[:, 0] > 0
mean_unique = len(torch.unique(mean_neighbors[active]))
sum_unique = len(torch.unique((adjacency @ ones)[active]))
assert mean_unique == 1 and sum_unique > mean_unique
print({"mean_unique": mean_unique, "sum_unique": sum_unique})

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §4.2, Eq. (4.1) / §4.3, Eq. (4.2)
- **이 셀의 책임:** GIN update / layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** epsilon self term / permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.ModuleList,nn.Sequential,nn.Linear,nn.ReLU,torch.stack,Tensor.detach -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 6개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.ModuleList(modules=None)</code></summary>

#### `nn.ModuleList(modules=None)`

- **역할:** 하위 모듈 목록을 등록 가능한 형태로 보관합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)

</details>

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
mlps = nn.ModuleList(
    [
        nn.Sequential(
            nn.Linear(features.shape[1] if i == 0 else 10, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
        )
        for i in range(3)
    ]
)
states = [features]
h = features
for mlp in mlps:
    h = mlp(gin_aggregate(h, adjacency, 0.1))
    states.append(h)
readouts = torch.stack([s.sum(0).norm() for s in states])
perm = torch.randperm(len(features))
p_adj = adjacency[perm][:, perm]
p_x = features[perm]
p_states = [p_x]
ph = p_x
for mlp in mlps:
    ph = mlp(gin_aggregate(ph, p_adj, 0.1))
    p_states.append(ph)
p_readouts = torch.stack([s.sum(0).norm() for s in p_states])
assert torch.allclose(readouts, p_readouts, atol=1e-5)
plt.plot(range(4), readouts.detach(), marker="o")
plt.xlabel("layer")
plt.ylabel("readout norm")
plt.title("GIN layer-wise readout")
plt.show()

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §4.2, Eq. (4.1) / §4.3, Eq. (4.2)
- **이 셀의 책임:** GIN update / layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** epsilon self term / permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,torch.where,torch.cat,nn.Parameter,F.cross_entropy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>torch.where(condition, input, other)</code></summary>

#### `torch.where(condition, input, other)`

- **역할:** 조건에 따라 두 텐서의 원소를 선택합니다.
- **입력·반환:** 불리언 조건과 두 입력을 받고 broadcast된 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** mask 기반 분기와 piecewise 수식을 벡터화합니다.
- **주의점:** 세 입력의 broadcasting과 두 branch의 dtype·device를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\begin{cases}x_i,&c_i=\mathrm{True}\\z_i,&c_i=\mathrm{False}\end{cases}
$$

- **기호:** $c_i$는 조건, $x_i,z_i$는 두 후보 값입니다.
- **수식 → 코드:** API가 Python 분기 대신 원소별로 두 tensor 중 하나를 선택합니다.
- **직관:** piecewise 수식과 mask 기반 update를 병렬 tensor 연산으로 표현합니다.
- **shape 확인:** 세 입력을 broadcasting한 공통 shape이 출력 shape입니다.
- **공식 문서:** [torch.where](https://docs.pytorch.org/docs/stable/generated/torch.where.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>nn.Parameter(data, requires_grad=True)</code></summary>

#### `nn.Parameter(data, requires_grad=True)`

- **역할:** 텐서를 학습 파라미터로 등록합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.Parameter.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    num_layers: int = 2
    learning_rate: float = 0.03
    steps: int = 18


def preprocess_graph(graph: Tensor) -> Tensor:
    return (graph > 0).to(graph.dtype)


def build_ego_graphs(
    inputs: Tensor,
    graph: Tensor,
    center_index: Tensor,
) -> list[tuple[Tensor, Tensor]]:
    examples = []
    for center in center_index.tolist():
        neighbors = torch.where(graph[center] > 0)[0]
        nodes = torch.cat(
            [torch.tensor([center]), neighbors],
        ).unique(sorted=True)
        ego_inputs = inputs[nodes].clone()
        ego_graph = graph[nodes][:, nodes].clone()
        examples.append((ego_inputs, ego_graph))
    return examples


class GinPortfolioModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        class_count: int,
        config: PortfolioConfig,
    ):
        super().__init__()
        self.config = config
        if config.num_layers < 2:
            raise ValueError("GIN requires at least two message-passing layers")
        self.epsilons = nn.ParameterList(
            [nn.Parameter(torch.tensor(0.0)) for _ in range(config.num_layers)]
        )
        self.mlps = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(
                        input_dim if layer == 0 else config.hidden_dim,
                        config.hidden_dim,
                    ),
                    nn.ReLU(),
                    nn.Linear(config.hidden_dim, config.hidden_dim),
                    nn.ReLU(),
                )
                for layer in range(config.num_layers)
            ]
        )
        readout_dim = input_dim + config.num_layers * config.hidden_dim
        self.graph_classifier = nn.Linear(
            readout_dim,
            class_count,
        )

    def encode(self, inputs: Tensor, graph: Tensor) -> list[Tensor]:
        states = [inputs]
        hidden = inputs
        layer_parameters = zip(
            self.epsilons,
            self.mlps,
            strict=True,
        )
        for epsilon, mlp in layer_parameters:
            aggregated = gin_aggregate(hidden, graph, epsilon)
            hidden = mlp(aggregated)
            states.append(hidden)
        return states

    def graph_readout(self, inputs: Tensor, graph: Tensor) -> Tensor:
        layer_states = self.encode(inputs, graph)
        layer_readouts = [state.sum(dim=0) for state in layer_states]
        return torch.cat(layer_readouts, dim=0)

    def forward(self, inputs: Tensor, graph: Tensor) -> Tensor:
        readout = self.graph_readout(inputs, graph)
        return self.graph_classifier(readout).unsqueeze(0)

    def compute_loss(
        self,
        outputs: Tensor,
        targets: Tensor,
    ) -> Tensor:
        return F.cross_entropy(outputs, targets)

    def training_step(
        self,
        inputs: Tensor,
        graph: Tensor,
        targets: Tensor,
    ) -> Tensor:
        outputs = self(inputs, graph)
        return self.compute_loss(outputs, targets)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §4.2, Eq. (4.1)
- **이 셀의 책임:** GIN update
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** epsilon self term. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Module.train,Tensor.reshape,Optimizer.zero_grad,Tensor.backward,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def fit_portfolio_model(
    model: GinPortfolioModel,
    graph_examples: list[tuple[Tensor, Tensor]],
    graph_targets: Tensor,
) -> list[float]:
    assert len(graph_examples) == len(graph_targets)
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    model.train()
    for _ in range(model.config.steps):
        example_losses = []
        paired_examples = zip(
            graph_examples,
            graph_targets,
            strict=True,
        )
        for (inputs, graph), target in paired_examples:
            example_loss = model.training_step(
                inputs,
                graph,
                target.reshape(1),
            )
            example_losses.append(example_loss)
        loss = torch.stack(example_losses).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    assert all(epsilon.grad is not None for epsilon in model.epsilons)
    assert model.graph_classifier.weight.grad is not None
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4.3, Eq. (4.2)
- **이 셀의 책임:** layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Module.eval,torch.no_grad,Tensor.argmax,torch.arange,Tensor.cpu,Tensor.mean,ACCELERATOR.move,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def evaluate_portfolio_model(
    model: GinPortfolioModel,
    graph_examples: list[tuple[Tensor, Tensor]],
    graph_targets: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        graph_logits = torch.cat(
            [model(inputs, graph) for inputs, graph in graph_examples],
            dim=0,
        )
        prediction = graph_logits.argmax(dim=1)
        accuracy = (prediction == graph_targets).float().mean()

        probe_inputs, probe_graph = graph_examples[0]
        permutation = torch.arange(
            len(probe_inputs) - 1,
            -1,
            -1,
            device=probe_inputs.device,
        )
        permuted_inputs = probe_inputs[permutation]
        permuted_graph = probe_graph[permutation][:, permutation]
        readout = model.graph_readout(probe_inputs, probe_graph)
        permuted_readout = model.graph_readout(
            permuted_inputs,
            permuted_graph,
        )
        permutation_error = (readout - permuted_readout).abs().max()
        assert torch.allclose(
            readout,
            permuted_readout,
            atol=1e-5,
        )
        assert torch.allclose(
            model(probe_inputs, probe_graph),
            model(permuted_inputs, permuted_graph),
            atol=1e-5,
        )

        layer_states = model.encode(probe_inputs, probe_graph)
        layer_norms = torch.stack([state.sum(dim=0).norm() for state in layer_states])

        degree_probe_graph = torch.tensor(
            [
                [0.0, 1.0, 0.0, 0.0],
                [1.0, 0.0, 1.0, 0.0],
                [0.0, 1.0, 0.0, 1.0],
                [0.0, 0.0, 1.0, 0.0],
            ],
            device=probe_inputs.device,
        )
        degree_probe_features = torch.ones(
            4,
            1,
            device=probe_inputs.device,
        )
        degree = degree_probe_graph.sum(dim=1, keepdim=True)
        mean_multiset = degree_probe_graph @ degree_probe_features
        mean_multiset = mean_multiset / degree.clamp_min(1.0)
        sum_multiset = gin_aggregate(
            degree_probe_features,
            degree_probe_graph,
            epsilon=0.0,
        )
        mean_unique = torch.unique(mean_multiset).numel()
        sum_unique = torch.unique(sum_multiset).numel()
        assert mean_unique == 1
        assert sum_unique > mean_unique
        assert torch.allclose(sum_multiset[:, 0], degree[:, 0] + 1.0)
    return {
        "heldout_ego_graph_accuracy": float(accuracy.cpu()),
        "readout_permutation_error": float(permutation_error.cpu()),
        "layer_readout_norm_mean": float(layer_norms.mean().cpu()),
        "layer_readout_count": float(len(layer_states)),
        "degree_multiset_unique_count": float(sum_unique),
    }


portfolio_train_index = torch.where(train_mask)[0]
portfolio_test_index = torch.where(test_mask)[0]
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_config = PortfolioConfig()
portfolio_graph = preprocess_graph(adjacency)
portfolio_train_graphs = build_ego_graphs(
    features,
    portfolio_graph,
    portfolio_train_index,
)
portfolio_test_graphs = build_ego_graphs(
    features,
    portfolio_graph,
    portfolio_test_index,
)
portfolio_train_graphs = [
    ACCELERATOR.move(ego_inputs, ego_graph)
    for ego_inputs, ego_graph in portfolio_train_graphs
]
portfolio_test_graphs = [
    ACCELERATOR.move(ego_inputs, ego_graph)
    for ego_inputs, ego_graph in portfolio_test_graphs
]
portfolio_train_targets, portfolio_test_targets = ACCELERATOR.move(
    labels[portfolio_train_index],
    labels[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(
    GinPortfolioModel(
        features.shape[1],
        int(labels.max()) + 1,
        portfolio_config,
    )
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_graphs,
    portfolio_train_targets,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_graphs,
    portfolio_test_targets,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["heldout_ego_graph_accuracy"] <= 1.0
assert portfolio_metrics["readout_permutation_error"] <= 1e-5
assert portfolio_metrics["layer_readout_count"] == portfolio_config.num_layers + 1
expected_readout_dim = features.shape[1]
expected_readout_dim += portfolio_config.num_layers * portfolio_config.hidden_dim
assert portfolio_model.graph_classifier.in_features == expected_readout_dim
assert portfolio_metrics["degree_multiset_unique_count"] > 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 9개 graph-classification benchmark 대신 한 로컬 그래프의 node multiset 표현력을 점검합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.